In [33]:
from secret_envs_wrapper import SecretEnv3
from collections import defaultdict
import sys
import numpy as np
sys.path.append("..")  # Va chercher un dossier au-dessus

In [34]:
env = SecretEnv3()
S = list(range(env.num_states()))
A = list(range(env.num_actions()))

R = [env.reward(i) for i in range(env.num_rewards())]

print("nombre d'etat :", env.num_states())
print("nombre d'actions :", env.num_actions())
print("nombre de rewards :", env.num_rewards())
for i in range (env.num_rewards()): 
    print("rewards{i}:",env.num_rewards())

nombre d'etat : 65536
nombre d'actions : 3
nombre de rewards : 3
rewards{i}: 3
rewards{i}: 3
rewards{i}: 3


In [23]:
# === Étape 1 : Exploration de l'environnement par simulation ===
def explorer_env_par_simulation(env, nb_episodes=10, max_steps=100):
    S = set()
    A = list(range(env.num_actions()))
    R = [env.reward(i) for i in range(env.num_rewards())]
    model = defaultdict(lambda: defaultdict(list))
    valid_actions_dict = defaultdict(set)
    
    for _ in range(nb_episodes):
        env.reset()
        for _ in range(max_steps):
            s = env.state_id()
            S.add(s)
            actions = env.available_actions()
            for a in actions:
                valid_actions_dict[s].add(a)
                for s_p in range(env.num_states()):
                    for r_idx, r in enumerate(R):
                        prob = env.p(s, a, s_p, r_idx)
                        if prob > 0:
                            model[s][a].append((prob, s_p, r))
            if len(actions) == 0:
                break
            action = np.random.choice(actions)
            env.step(action)
            if env.is_game_over():
                break

    return sorted(S), A, R, model, {k: list(v) for k, v in valid_actions_dict.items()}

In [31]:
def iterative_policy_evaluation_sparse(
    pi: np.ndarray,
    S: list,
    A: list,
    model: dict,
    terminal_states: list,
    valid_actions_dict: dict,
    gamma: float = 0.8,
    theta: float = 1e-2,
    max_iter: int = 10  # Limite d'itérations
):
    V = np.zeros(max(S)+1)
    iteration = 0

    while iteration < max_iter:
        delta = 0.0
        iteration += 1

        for s in S:
            if s in terminal_states:
                continue
            v = V[s]
            total = 0.0
            for a in valid_actions_dict.get(s, A):
                for (prob, s_p, r) in model[s][a]:
                    total += pi[s, a] * prob * (r + gamma * V[s_p])
            V[s] = total
            delta = max(delta, abs(v - V[s]))

        print(f"[Itération {iteration}] delta = {delta:.6f}")

        if delta < theta:
            print("✔️ Convergence atteinte.")
            break

    print("\n🔍 Aperçu des 20 premiers états :")
    for s in sorted(S)[:20]:
        print(f"État {s} : V = {V[s]:.4f}")

    return V


In [35]:
import numpy as np
import time
import random

def iterative_policy_evaluation_sparse(
    pi: np.ndarray,
    S: list,
    A: list,
    model: dict,
    terminal_states: list,
    valid_actions_dict: dict,
    gamma: float = 0.8,
    theta: float = 1e-2,
    max_iter: int = 10  # Limite d'itérations
):
    V = np.zeros(max(S) + 1)
    iteration = 0

    while iteration < max_iter:
        start_time = time.time()
        delta = 0.0
        iteration += 1

        for s in S:
            if s in terminal_states:
                continue
            v = V[s]
            total = 0.0
            for a in valid_actions_dict.get(s, A):
                for (prob, s_p, r) in model[s][a]:
                    total += pi[s, a] * prob * (r + gamma * V[s_p])
            V[s] = total
            delta = max(delta, abs(v - V[s]))

        print(f"[Itération {iteration}] delta = {delta:.6f} – durée : {time.time() - start_time:.2f} sec")

        if delta < theta:
            print("✔️ Convergence atteinte.")
            break

    print("\n🔍 Aperçu des 20 premiers états :")
    for s in sorted(S)[:20]:
        print(f"État {s} : V = {V[s]:.4f}")

    return V

# === MAIN TEST ===
if __name__ == "__main__":
   

    env = SecretEnv3()
    print("Exploration de l'environnement en cours...")
    S, A, R, model, valid_actions_dict = explorer_env_par_simulation(env)

    print(f"États explorés : {len(S)} / {env.num_states()}")
    print(f"Récompenses détectées : {R}")

    # 💡 Limitation temporaire pour test rapide
    if len(S) > 2000:
        S = random.sample(S, 2000)

    # Politique uniforme sur actions valides
    pi = np.zeros((max(S) + 1, len(A)))
    for s in S:
        actions = valid_actions_dict.get(s, [])
        if actions:
            for a in actions:
                pi[s, a] = 1.0 / len(actions)

    terminal_states = [s for s in S if not valid_actions_dict.get(s)]

    V = iterative_policy_evaluation_sparse(
        pi=pi,
        S=S,
        A=A,
        model=model,
        terminal_states=terminal_states,
        valid_actions_dict=valid_actions_dict,
        gamma=0.8,
        theta=1e-2,
        max_iter=10
    )


Exploration de l'environnement en cours...


KeyboardInterrupt: 